# Refugio Animal – versión 3: arquitectura y código

Análisis detallado de la estructura del proyecto, la responsabilidad de cada fichero y el flujo de ejecución.

Esta versión incorpora la **gestión dinámica de especies** desde Tkinter y una **ficha informativa** que utiliza realmente el modelo POO. Además, el **inventario de animales** se persiste en JSON (separado del catálogo de especies).

## Distinción arquitectónica clave

- El inventario de animales utiliza **diccionarios** (+ persistencia `inventario.json`).
- Las especies **nativas** se representan mediante **clases Python**.
- Las especies **personalizadas** usan `EspeciePersonalizada` configurada con datos.
- El **adaptador** conecta registros del inventario con objetos POO.
- **Tkinter** usa esos objetos para mostrar información y comportamientos.

Es una arquitectura **híbrida**: programación estructurada + POO + persistencia JSON.

## 1. Estructura general de carpetas

```text
entregable_tarea1/
│
├── gestion_animales_tkinter.ipynb
├── arquitectura_refugio_animal_v3.ipynb
│
└── tarea1/
    ├── app.py
    ├── main.py
    ├── ejecutar_tests.py
    ├── probar_crud.py
    ├── pyproject.toml
    ├── .python-version
    ├── README.md
    ├── refugio.jpg
    ├── Enunciado_oficial_Tarea_1.pdf
    ├── adjuntos/
    ├── datos/
    ├── docs/
    ├── exportacion/
    ├── interfaz/
    ├── muestras/
    ├── tests/
    └── validaciones/
```

Cada carpeta agrupa módulos por **responsabilidad**.

## 2. Carpeta raíz `tarea1/`

| Archivo | Función |
| --- | --- |
| `app.py` | Punto de entrada: carga inventario, crea Tkinter e inicia la app |
| `main.py` | Alternativa de arranque que delega en `app.py` |
| `ejecutar_tests.py` | Ejecuta pruebas y genera informes |
| `probar_crud.py` | Comprueba el CRUD sin GUI |
| `pyproject.toml` | Metadatos, Python y dependencias |
| `.python-version` | Python 3.14 |
| `README.md` | Instalación, ejecución y estructura |
| `refugio.jpg` | Logo de la aplicación |
| `Enunciado_oficial_Tarea_1.pdf` | Enunciado del ejercicio |

`app.py` es deliberadamente pequeño: solo inicia, no contiene la lógica de negocio.

In [ ]:
# Esquema del arranque (equivalente a app.py)
from pathlib import Path
import sys

RAIZ_PROGRAMA = Path("tarea1").resolve()  # en la app real: Path(__file__).parent
if str(RAIZ_PROGRAMA) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROGRAMA))

print("1. Se añade la carpeta del programa a sys.path")
print("2. cargar_inventario() lee o siembra datos/inventario.json")
print("3. tk.Tk() crea la ventana")
print("4. GestorAnimales(raiz, inventario) construye la UI")
print("5. mainloop() espera eventos del usuario")

### Instrucciones clave de `app.py`

1. `Path(__file__).resolve().parent` obtiene la carpeta de `app.py`.
2. Se añade esa ruta a `sys.path` para facilitar los imports.
3. `cargar_inventario()` carga (o siembra) `datos/inventario.json`.
4. `tk.Tk()` crea la ventana principal.
5. `GestorAnimales()` construye la interfaz con el inventario.
6. `mainloop()` inicia el bucle de eventos.

`main.py` solo hace `from app import main` y llama a `main()`.

## 2.2. Carpeta `datos/` — núcleo de la aplicación

| Archivo | Responsabilidad |
| --- | --- |
| `__init__.py` | Identifica el paquete |
| `constantes.py` | Especies nativas y límites de peso |
| `inventario.py` | Operaciones CRUD sobre diccionarios |
| `persistencia_inventario.py` | Carga/guarda `inventario.json` (atómico) |
| `modelos.py` | Clases POO |
| `decoradores.py` | Decorador `cuenta_usos` |
| `adaptador_poo.py` | Dict ↔ objeto POO |
| `gestor_especies.py` | Catálogo dinámico de especies |
| `especies_personalizadas.json` | Especies creadas desde Tkinter |
| `inventario.json` | Animales persistidos (se crea al arrancar) |
| `ficha.py` | Tratamientos, vacunas, documentos, foto |
| `adjuntos.py` | Archivos físicos por animal |
| `recursos.py` | Localiza el logo |

**Separación importante:** el catálogo de especies y el inventario de animales son dos JSON distintos.

### A. `constantes.py`

Define las cinco especies nativas (`Perro`, `Gato`, `Conejo`, `Loro`, `Hámster`) y los límites de peso. Las personalizadas **no** se añaden aquí: las gestiona `gestor_especies.py`.

In [ ]:
# Constantes nativas (resumen)
ESPECIES = ("Perro", "Gato", "Conejo", "Loro", "Hámster")
PESO_MIN_KG = 0.01
PESO_MAX_KG = 100.0
print(ESPECIES)

### B. `inventario.py`

Tipo: `Inventario = Dict[int, Dict[str, Any]]`.

| Función | CRUD | Responsabilidad |
| --- | --- | --- |
| `animales_iniciales()` | Init | Animales de ejemplo |
| `crear_animal()` | Create | Valida y registra |
| `leer_animal()` | Read | Copia del registro |
| `actualizar_animal()` | Update | Modifica datos |
| `borrar_animal()` | Delete | Elimina animal y adjuntos |

`crear_animal()` **no** es una función de Tkinter: recibe datos, valida y modifica el inventario. Así se puede probar sin GUI.

In [ ]:
# Forma del inventario (ejemplo)
inventario = {
    1: {
        "nombre": "Luna",
        "especie": "Perro",
        "edad": 4,
        "peso": 18.5,
        "color": "Canela",
        "observaciones": "Sociable",
        "foto": None,
        "tratamientos": [],
        "vacunaciones": [],
        "documentos": [],
    }
}
print("Clave = ID; valor = diccionario de campos")

### Persistencia del inventario (`persistencia_inventario.py`)

Novedad respecto al análisis anterior (solo especies en JSON):

- `cargar_inventario()`: si no existe el archivo, siembra `animales_iniciales()` y lo guarda.
- `guardar_inventario()`: escritura atómica (temporal + `replace()`).
- Tras crear / actualizar / borrar / cambiar ficha clínica, `GestorAnimales` llama a `guardar_inventario()`.
- El catálogo sigue en `especies_personalizadas.json` (independiente).

In [ ]:
# Demostración de la API de persistencia (ruta temporal)
import sys
from pathlib import Path
import tempfile

raiz = Path.cwd() / "tarea1"
if not (raiz / "datos").is_dir():
    raiz = Path.cwd()
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from datos.persistencia_inventario import cargar_inventario, guardar_inventario
from datos.inventario import crear_animal

with tempfile.TemporaryDirectory() as tmp:
    ruta = Path(tmp) / "inventario.json"
    inv = cargar_inventario(ruta)
    crear_animal(
        {
            "nombre": "Nico",
            "especie": "Perro",
            "edad": 2,
            "peso": 1.2,
            "color": "Marrón",
            "observaciones": "Demo persistencia",
        },
        inv,
    )
    guardar_inventario(inv, ruta)
    otra = cargar_inventario(ruta)
    print("Animales tras recargar:", sorted(otra))
    print("Nuevo:", otra[max(otra)]["nombre"])

### C. `modelos.py` — POO

Jerarquía:

```text
AnimalRefugio (ABC)
├── Perro / Gato / Conejo / Loro / Hámster
└── EspeciePersonalizada  (Hurón, Tortuga, … vía configuración)
```

- **Encapsulación:** `@property` + setters (edad 0–30, peso, nombre…).
- **Abstracción:** `@abstractmethod` para `hacer_sonido`, `tipo_alimentacion`, `cuidados_especificos`.
- **Polimorfismo:** `generar_ficha()` llama esos métodos; cada clase responde distinto.
- **`EspeciePersonalizada`:** no es una clase nueva por especie; recibe sonido/alimentación/cuidados por parámetros.

In [ ]:
# Polimorfismo con generar_ficha (si el paquete está disponible)
import sys
from pathlib import Path

raiz = Path.cwd() / "tarea1"
if not (raiz / "datos").is_dir():
    raiz = Path.cwd()
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from datos.modelos import Perro, EspeciePersonalizada

luna = Perro("Luna", 4, 18.5, "Canela", "Sociable")
nico = EspeciePersonalizada(
    "Nico", 2, 1.2, "Marrón", "Curioso",
    nombre_especie="Hurón",
    sonido="Emite pequeños chillidos",
    alimentacion="Carnívoro",
    cuidados="Entorno seguro y supervisión",
)
for animal in (luna, nico):
    ficha = animal.generar_ficha()
    print("---", animal.especie, animal.nombre)
    print(ficha["comportamiento"])
    print(ficha["alimentacion"])

### D. `decoradores.py`

`@cuenta_usos` incrementa un contador `_usos_<metodo>` en el objeto cada vez que se llama (p. ej. `info_basica`). Demuestra ampliar comportamiento sin tocar la lógica original.

### E. `adaptador_poo.py`

| Función | Responsabilidad |
| --- | --- |
| `registro_a_animal()` | Dict → objeto (nativa o personalizada) |
| `animal_a_campos()` | Objeto → campos básicos |
| `resumen_polimorfico()` | Resumen vía métodos del objeto |

Flujo de `registro_a_animal()`: especie → clase nativa si existe → si no, config del catálogo → `EspeciePersonalizada` → error controlado si falla.

### F. `gestor_especies.py` — `CatalogoEspecies`

Persistencia / consultas / modificaciones del JSON de especies. La interfaz no escribe JSON a mano: llama `catalogo.anadir(datos)`.

### G–I. `ficha.py`, `adjuntos.py`, `recursos.py`

Clínica homogénea, archivos bajo `adjuntos/<id>/fotos|documentos/` (límite 5 MB) y localización del logo.

## 2.3. `validaciones/`

`validar_datos(datos, especie_previa=None)` comprueba nombre, especie activa, color, edad, peso y estructuras clínicas. Consulta el catálogo dinámico (`especie_permitida_en_alta`). Con `especie_previa` se pueden conservar animales de especies desactivadas.

La validación es **lógica de dominio**, no del botón de Tkinter.

## 2.4. `interfaz/`

| Archivo | Responsabilidad |
| --- | --- |
| `gestor.py` | Ventana principal y eventos |
| `dialogo_especies.py` | Gestionar especies (`Toplevel`) |
| `ficha_clinica.py` | Tratamientos, vacunas, docs, fotos |
| `estilos.py` | Tema y tipografía |
| `vista_foto.py` | Miniaturas |
| `portapapeles.py` | Ctrl+V (archivos / imagen / texto) |

### Ficha POO en `gestor.py`

Al seleccionar una fila: `registro_a_animal(registro)` → `animal.generar_ficha()` → paneles Tkinter. La UI **no** pregunta si es perro o hurón.

Tras CRUD o cambios clínicos: `_persistir_inventario()` escribe `inventario.json`.

## 2.5–2.9. Otras carpetas

| Carpeta | Rol |
| --- | --- |
| `exportacion/` | PDF con `fpdf2` (`PdfFichas`) desde diccionarios |
| `muestras/` | Fotos/docs de prueba (`generar_muestras.py`) |
| `adjuntos/` | Archivos importados por el usuario |
| `tests/` | Unit tests + `resultados/` |
| `docs/` | Requisitos, casos de uso, manuales, UML, este PDF |

## 3. Flujos de ejecución

### 3.1 Arranque

`app.py` → `cargar_inventario()` → `GestorAnimales` → `mainloop()`

### 3.2 Alta de animal

Formulario → `_crear()` → `crear_animal()` → `validar_datos()` + catálogo → inventario → **`guardar_inventario()`** → tabla

### 3.3 Selección y POO

Treeview → dict → `registro_a_animal()` → `generar_ficha()` → panel Tkinter

### 3.4 Especie personalizada

*Gestionar especies* → `CatalogoEspecies.anadir()` → `especies_personalizadas.json` → refrescar combobox → alta de animales de esa especie

## 4. Conceptos POO demostrados

| Concepto | Aplicación |
| --- | --- |
| Clases y objetos | `AnimalRefugio`, nativas, `EspeciePersonalizada` |
| Abstracción | ABC + métodos abstractos |
| Herencia | Especies concretas |
| Encapsulación | Properties / setters |
| Polimorfismo | `generar_ficha()` / sonido / cuidados |
| Decoradores | `@cuenta_usos` |
| Métodos de clase | `campos_desde_dict()` |

Una especie personalizada **no** es una clase Python nueva: es un `EspeciePersonalizada` configurado (p. ej. como Hurón).

## 5. Dependencias y ejecución

```text
requires-python >= 3.14
dependencies: fpdf2>=2.9.0
Pillow opcional (imágenes / portapapeles)
```

```bash
cd tarea1
python app.py
python ejecutar_tests.py
# o: python -m unittest discover -s tests -v
```

Con uv: `uv sync` y `uv run python app.py`.

## 6. Cuadernos Jupyter

| Característica | `gestion_animales_tkinter.ipynb` | App `tarea1/` | Este cuaderno |
| --- | --- | --- | --- |
| Rol | Demo didáctica CRUD/POO | Entrega completa | Documentación de arquitectura |
| Especies JSON | Demo | Completo | Explica |
| Persistencia inventario | No | Sí (`inventario.json`) | Explica + demo |
| Fotos / clínica / PDF | No | Sí | Describe |

## 7–9. Observaciones y evolución

1. **Persistencia:** especies en un JSON; animales en otro (`inventario.json`). Los adjuntos siguen en carpetas por ID.
2. **Acentos:** `Hámster` ≠ `Hamster` (casefold no quita tildes).
3. **README:** debe reflejar solo ficheros presentes en la entrega.
4. **Pruebas GUI:** requieren entorno gráfico Windows.

| Versión | Característica |
| --- | --- |
| 1 | CRUD + diccionarios + Tkinter |
| 2 | POO, adaptador, ficha polimórfica |
| 3 | Especies dinámicas + JSON + ficha POO completa |
| 3+ | Persistencia del inventario de animales |

**Recorrido recomendado de demo:** crear especie Hurón → registrar Nico → seleccionar → explicar adaptador + `EspeciePersonalizada` + `generar_ficha()` → cerrar y reabrir la app para mostrar que el inventario persistió.